In [37]:
import pandas as pd
from pathlib import Path
from datetime import datetime, date
import ast

In [38]:
def ensure_columns(df, fill_value=pd.NA):

    required_columns = ['json_name', 'column_name', 'path',
                        'list_path', 'subfield_path', 'var_type', 
                        'data_type', 'file_path', 'id', 'keepID']
    
    for col in required_columns:
        if col not in df.columns:
            df[col] = fill_value

    return df

In [39]:
def combine_list(df):

    df['final_path'] = None

    for ix, row in df.iterrows():
        if row['platform'] == 'Tiktok':
            final_path =  row['path']
            
            if isinstance(final_path, str):
                final_path = ast.literal_eval(final_path)
               
            
        else:
            file_list = row['file_path'].split('/')
            path_list = row['path']
        
            if pd.isna(path_list):
                final_path = row['file_path']
            else:
                if isinstance(path_list, str):
                    path_list = ast.literal_eval(path_list)

                if isinstance(path_list, list):
                    final_path = file_list + path_list
                    path = '/'.join(path_list)
                    df.at[ix, 'path'] = path

        #print(final_path)
        final_path = '/'.join(final_path)
        df.at[ix, 'final_path'] = final_path 
              
    
    return df
    


In [40]:

#ROOT = '/home/rvissche/GIT/social-media-data-map'
ROOT = '/home/bsc/bsc093754/GIT/social-media-data-map'
root_dir = Path(f"{ROOT}/data/raw/annotated_merged_structures")
save_dir = f'{ROOT}/data/processed/'
  

def create_dataset(root_dir, save_dir):

    dfs_list = []

    for csv_file in root_dir.glob("*.csv"):

        try:
            file_name =  csv_file.stem
            print(f'{datetime.now()} PROCESSING : {file_name}')
            
            df = pd.read_csv(csv_file)
            print(f'{datetime.now()} READ CSV : {file_name}')
            

            if 'TT_' in str(csv_file):
                df['platform'] = 'Tiktok'
            if 'IG_' in str(csv_file):
                df['platform'] = 'Instagram'
            if 'FB_' in str(csv_file):
                df['platform'] = 'Facebook'
            if 'YT_' in str(csv_file):
                df['platform'] = 'Youtube'
            if 'X_' in str(csv_file):
                df['platform'] = 'Twitter'

            df = combine_list(df)
            print(f'{datetime.now()} FINISH COMBINE LIST : {file_name}')
            
            col = df.pop('platform') 
            df.insert(1, 'platform', col) 

            df = ensure_columns(df)
            print(f'{datetime.now()} FINISH ENSURE COLUMNS: {file_name}')
            df = df.dropna(subset=['keepID'])
            
            dfs_list.append(df)
            print(f'{datetime.now()} FINISH APPEND {file_name}')
            print('FILE_NAME done: ', file_name)
        
        except Exception as e:
            print(f"Failed to load {csv_file}: {e}")


    dfs = pd.concat(dfs_list, ignore_index=True)




    print('DATASET CREATED at ', datetime.now())
    dfs.to_csv(f'{save_dir}/annotated_paths{date.today()}.csv')
    print('DATASET SAVED TO ', save_dir)

    return dfs

df = create_dataset(root_dir, save_dir)



2026-07-28 15:19:39.387506 PROCESSING : X_merged_structure_annotated
2026-07-28 15:19:39.390393 READ CSV : X_merged_structure_annotated
2026-07-28 15:19:39.438606 FINISH COMBINE LIST : X_merged_structure_annotated
2026-07-28 15:19:39.439046 FINISH ENSURE COLUMNS: X_merged_structure_annotated
2026-07-28 15:19:39.440129 FINISH APPEND X_merged_structure_annotated
FILE_NAME done:  X_merged_structure_annotated
2026-07-28 15:19:39.440183 PROCESSING : FB_merged_structure_annotated
2026-07-28 15:19:39.452492 READ CSV : FB_merged_structure_annotated
2026-07-28 15:19:39.749214 FINISH COMBINE LIST : FB_merged_structure_annotated
2026-07-28 15:19:39.749787 FINISH ENSURE COLUMNS: FB_merged_structure_annotated
2026-07-28 15:19:39.751488 FINISH APPEND FB_merged_structure_annotated
FILE_NAME done:  FB_merged_structure_annotated
2026-07-28 15:19:39.751563 PROCESSING : YT_merged_column_names_annotated
2026-07-28 15:19:39.752974 READ CSV : YT_merged_column_names_annotated
Failed to load /home/bsc/bsc0937

In [41]:
def inference_sample(df):
    
    new_df = df[df["id"] != df["keepID"]].copy()
    new_df = new_df[['platform','final_path', 'id', 'keepID']]
    new_df.to_csv(f'{save_dir}/inference_sample{date.today()}.csv')
    return new_df

inference_sample(df)


         

,platform,final_path,id,keepID
182,Facebook,preferences/feed/snooze.json/media,preferences:feed:snooze:media,feed:snooze:media
183,Facebook,preferences/feed/snooze.json/fbid,preferences:feed:snooze:fbid,feed:snooze:fbid
186,Facebook,your_facebook_activity/other_activity/your_vid...,your_facebook_activity:other_activity:your_vid...,other_activity:your_video_consumption_summary:...
187,Facebook,your_facebook_activity/other_activity/your_vid...,your_facebook_activity:other_activity:your_vid...,other_activity:your_video_consumption_summary:...
221,Facebook,your_facebook_activity/posts/edits_you_made_to...,your_facebook_activity:posts:edits_you_made_to...,posts:edits_you_made_to_posts:timestamp
...,...,...,...,...
1974,Tiktok,Your Activity/Like List/ItemFavoriteList/date,Your Activity:Like List:ItemFavoriteList:date,Activity:Like List:ItemFavoriteList:date
1975,Tiktok,Your Activity/Like List/ItemFavoriteList/link,Your Activity:Like List:ItemFavoriteList:link,Activity:Like List:ItemFavoriteList:link
1988,Tiktok,Your Activity/Share History/ShareHistoryList/Date,Your Activity:Share History:ShareHistoryList:Date,Activity:Share History:ShareHistoryList:Date
1989,Tiktok,Your Activity/Share History/ShareHistoryList/S...,Your Activity:Share History:ShareHistoryList:S...,Activity:Share History:ShareHistoryList:Shared...
